# Entrena un modelo de reconocimiento de imágenes con Python

Tiempo estimado: **40** minutos

¿Alguna vez has querido crear un detector de imágenes que te indique si una imagen contiene un objeto determinado o no?

Históricamente, los científicos de la computación han adoptado dos enfoques principales para determinar si una imagen contiene, por ejemplo, un perrito caliente (hot dog). Estos representan dos corrientes fundamentales de la inteligencia artificial. La primera consistía en programar en un sistema informático el conocimiento necesario para reconocer el contenido de una imagen, permitiendo que la máquina utilizara reglas de inferencia de forma similar a como lo hacen los seres humanos. La segunda consistía en construir una gran red de neuronas capaz de aprender a partir de un extenso conjunto de datos del mundo real mediante una regla de aprendizaje **sencilla**. Este último enfoque ha cosechado un gran éxito y hoy se conoce como «aprendizaje profundo» (*deep learning*); por el contrario, la primera opción es considerada actualmente por la mayoría de los expertos como una idea biológicamente poco realista.

En este proyecto, utilizaremos esas reglas de aprendizaje sencillas para «enseñar» a una red neuronal a reconocer perritos calientes en una imagen y resolver la gran incógnita: ¿es un perrito caliente o no?

¡Vamos a descubrirlo!

## Qué necesitarás

Antes de comenzar este laboratorio, te resultará útil estar familiarizado con lo siguiente:

*   Conocimientos básicos de Python
*   Conocimientos básicos sobre clasificación de imágenes y redes neuronales, que se tratan en las lecturas opcionales

## Lo que aprenderás

Tras completar este laboratorio, serás capaz de:

* Crear funciones, utilizando bibliotecas y paquetes, para previsualizar, cargar y entrenar tu modelo.
* Transformar conjuntos de datos para prepararlos para el entrenamiento.
* Entrenar un clasificador de imágenes de última generación mediante aprendizaje por transferencia (*transfer learning*).

## Ejercicio 1: ¡Prepara tu kit de herramientas de Python!<a id="Exercise1"></a>

¿Qué debería tener todo usuario de Python en su kit de herramientas? Su conjunto de datos, bibliotecas de Python y algunas funciones personalizadas que faciliten el proceso de clasificación de imágenes. ¡Echa un vistazo!

### Tarea A: Configuración del entorno<a id="Task1A"></a>

Antes de comenzar, vamos a configurar tu entorno. Empezaremos actualizando PyTorch —una biblioteca de Python para aprendizaje profundo— en este cuaderno.

In [ ]:
%pip install torch torchvision torchaudio --index-url https://pytorch.org



Ahora, procedamos a comprobar si este entorno cuenta con un modelo de GPU compatible o no:

In [ ]:
from torch.cuda import is_available, get_device_name

if is_available():
    print(f"The environment has a compatible GPU ({get_device_name()}) available.")
else:
    print(f"The environment does NOT have a compatible GPU model available.")

### Tarea B: Configuración del conjunto de datos<a id="Task1B"></a>

Dado que las redes deben aprender a partir de datos del mundo real, el primer paso consiste en recopilar un gran conjunto de imágenes y etiquetar cada una de ellas como «hotdog» (perrito caliente) o «not hotdog» (no es un perrito caliente).

En este proyecto, hemos preparado el conjunto de datos con sus etiquetas; puedes acceder a él a través de este enlace: [Conjunto de datos «Hotdog Not Hotdog»](URL). No es necesario que descargues el conjunto de datos y lo subas manualmente a nuestro entorno de laboratorio, ya que nuestro cargador de datos se encargará de ello por ti.

El conjunto de datos que proporcionamos contiene la correspondencia entre cada imagen y su etiqueta («hotdog» o «not hotdog») dentro de un archivo `csv`. Este archivo incluye la siguiente información, que se utilizará más adelante en este laboratorio para entrenar el modelo: `label`, `y` y `file_name`. Para comprender mejor qué representa cada una de estas columnas, consulta la siguiente tabla:

| Columna    | Valores                   | Descripcion                                                                                                                       |
| --------- | ------------------------ | --------------------------------------------------------------------------------------------------------------------------------- |
| Label     | "hotdog" or "not_hotdog" | Describes whether or not a image is of a hotdog.                                                                                  |
| y         | "0" or "1"               | Numerical representation of the label, i.e. "0": denotes that an image is of a hotdog, "1": denotes the image is not of a hotdog. |
| file_name | Various Unique Values    | Denotes the path to the images, e.g. `hotdognothotdogfull/hotdog_1.jpg`.   

### Tarea C: Configuración de tus herramientas<a id="Task1C"></a>

Las bibliotecas de Python son de gran ayuda para simplificar y facilitar la programación. En lugar de escribir código desde cero, puedes utilizar funciones que ya han sido preescritas. Puedes obtener más información sobre las bibliotecas que hemos importado consultando la [documentación de la biblioteca estándar de Python](https://docs.python.org/3/library/index.html?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkQuickLabsbuildahotdognothotdogclassifierguidedproject28954781-2022-01-01).

¡Echemos un vistazo a las bibliotecas de Python que utilizaremos en este proyecto!

#### PyTorch: El framework de redes neuronales para aprendizaje profundo

[PyTorch](https://pytorch.org/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkQuickLabsbuildahotdognothotdogclassifierguidedproject28954781-2022-01-01) es una biblioteca de tensores optimizada para el aprendizaje profundo (*deep learning*) que utiliza GPU (unidades de procesamiento gráfico), CPU (unidades centrales de procesamiento) y CUDA (*Compute Unified Device Architecture*); fue desarrollada por el laboratorio de IA de Facebook. Se trata de un proyecto de código abierto creado sobre la base de una biblioteca de tensores.

PyTorch ofrece herramientas prácticas para el cálculo con arreglos mediante aceleración por GPU, así como métodos integrados para la construcción y el entrenamiento de modelos de redes neuronales.

#### PIL, Matplotlib y NumPy: Bibliotecas para el procesamiento de imágenes y la visualización de datos

Entre las bibliotecas más populares para el procesamiento y la visualización de datos se encuentran Python Imaging Library (PIL), utilizada para manipular diversos formatos de archivos de imagen; Matplotlib, para la creación de gráficos; NumPy, para el manejo de arreglos y matrices; y tqdm, empleada para mostrar barras de progreso a lo largo del laboratorio.

Puedes obtener más información sobre estas bibliotecas y sus aplicaciones en los cursos «Análisis de datos con Python» y «Visualización de datos con Python».

#### Funciones y clases: creación de utilidades

Ahora que ya conoces las librerías que utilizaremos, podemos crear clases y funciones auxiliares que nos ayuden a entrenar nuestro modelo. Antes de definirlas, importaremos las librerías correspondientes.

##### Visualización de imágenes

A lo largo de este laboratorio, necesitaremos visualizar las imágenes con las que contamos para saber qué tipo de datos tenemos. ¿Y qué mejor forma de hacerlo que creando una función específica para ello?

In [ ]:
from numpy import clip
from matplotlib import pyplot as plt
from torch import Tensor

def imshow(inp: Tensor) -> None:
    """Imshow for Tensor."""
    inp = inp.cpu().numpy()
    inp = inp.transpose((1, 2, 0))
    mean = array([0.485, 0.456, 0.406])
    std = array([0.229, 0.224, 0.225])
    inp = std * inp + mean
    inp = clip(inp, 0, 1)
    plt.imshow(inp)
    plt.show()

##### Carga de tu conjunto de datos

A continuación, crearemos una clase `HotdogDataset`. Esta clase descargará el conjunto de datos si es necesario, lo transformará según los parámetros que indiquemos y asociará cada imagen del conjunto de datos con una etiqueta: «hotdog» o «no es un hotdog».

In [ ]:
"""
Hotdog/Not Hotdog training data.
"""
import os
from typing import Tuple
from zipfile import ZipFile
from PIL import Image

from numpy import array
from pandas import read_csv
from requests import get
from torch.utils.data import Dataset


class HotDogDataset(Dataset):
    """
    Child class of torch.utils.data.Dataset.

    This is a wrapper for mapping from hotdog/not hotdog images to the target.
    """

    def __init__(self, dir_name, transform=None) -> None:
        """
        Initialise a HotdogDataset class

        :param dir_name: The name of the folder holding the data.
        :param transform:
        """
        self.transform = transform
        # Download, save and extract the data if needed.
        if not os.path.isdir(os.path.join(os.getcwd(), f"{dir_name}")):
            r = get(
                f"https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/hotdog-not-hotdog/data/{dir_name}.zip")
            f = open(os.path.join(os.getcwd(), f"{dir_name}.zip"), mode="wb+")
            f.write(r.content)
            f.close()
            with ZipFile(os.path.join(os.getcwd(), f"{dir_name}.zip"), 'r') as zip_ref:
                zip_ref.extractall("./")
                zip_ref.close()
        # Load the metadata.
        self.data = read_csv(os.path.join(os.getcwd(), dir_name, f"{dir_name}_labels.csv"))
        # Number of classes.
        self.n_classes = len(self.data['y'].unique())

    def __len__(self) -> int:
        """
        :return: The length of the training/testing dataset.
        """
        return len(self.data)

    def __getitem__(self, idx) -> Tuple[array, str]:
        """
        Return the input and target at a specific index of the dataset.

        :param idx: The index of the data to be returned.
        :return: Key-value pair at the specified index.
        """
        # Open the corresponding Image.
        image = Image.open(os.path.join(os.getcwd(), self.data.loc[idx, 'file_name']))
        # Retrieve the label.
        y = self.data.loc[idx, 'y']
        # Transform the image if necessary.
        if self.transform is not None:
            image_ = self.transform(image)
            image.close()
        else:
            image_ = array(image)
            image.close()
        return image_, y

##### Creación de un marco de entrenamiento

Por último, crearemos la clase `ClassificationModelTrainer`. Utilizaremos esta clase como marco de trabajo para entrenar, validar y probar el modelo. Una ventaja de crear este marco es que proporciona un método práctico para entrenar nuestro modelo con los hiperparámetros deseados, así como para visualizar los resultados y las métricas de entrenamiento que nos interesa observar. Hablaremos más sobre estas métricas más adelante en este proyecto.

In [ ]:
"""
A training framework for classification tasks.
"""

from copy import deepcopy
from typing import List, Tuple
from os.path import join
from os import getcwd

from matplotlib.pyplot import subplots, show
from numpy import sum
from torch import argmax, device, cuda, save, load
from torch.nn import Module
from torch.nn.functional import softmax
from torch.nn.modules.loss import _Loss
from torch.optim import Optimizer
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm



class ClassificationModelTrainer:

    def __init__(self,
                 model: Module,
                 training_set: Dataset,
                 validation_set: Dataset,
                 batch_size: int,
                 minimising_criterion: _Loss,
                 optimiser: Optimizer) -> None:
        """
        Initialise a classification model training module.

        :param model: The model to train.
        :param training_set: The set of training data.
        :param validation_set: The set of validation data.
        :param batch_size: The batch size for training.
        :param minimising_criterion: The loss function.
        :param optimiser: The algorithm to perform minimisation task.
        """
        self._device = device("cuda:0" if cuda.is_available() else "cpu")
        self._model = model.to(self._device)
        self._train_loader = DataLoader(dataset=training_set, batch_size=batch_size, shuffle=True)
        self._validation_loader = DataLoader(dataset=validation_set, batch_size=batch_size, shuffle=True)
        self._minimising_criterion = minimising_criterion
        self._optimiser = optimiser
        self.training_loss = []
        self.validation_acc = []

    def get_model(self) -> Module:
        """
        Getter function for model.

        :return: Return the trained model.
        """
        return self._model

    def train_model(self, n_epochs) -> None:
        """
        Perform the model training.

        :param n_epochs: The number of training epochs to run.
        """
        # Setup the progress bar.
        pbar = tqdm(total=n_epochs * (len(self._train_loader) + len(self._validation_loader)))
        pbar.set_postfix({
            "Training Loss": "Not yet available" if len(self.training_loss) == 0 else self.training_loss[-1],
            "Validation Accuracy": "Not yet available" if len(self.validation_acc) == 0 else self.validation_acc[-1],
            "Epoch": 1
        })
        # Training through the epochs.
        for epoch in range(n_epochs):
            loss_sublist = []
            # Training Process
            for x, y in self._train_loader:
                x, y = x.to(self._device), y.to(self._device)
                self._model.train()
                z = self._model(x)
                loss = self._minimising_criterion(z, y)
                loss_sublist.append(loss.data.item())
                loss.backward()
                self._optimiser.step()
                self._optimiser.zero_grad()
                pbar.update()
            self.training_loss.append(sum(loss_sublist))
            # Validation Process
            correct = 0
            n_test = 0
            for x_test, y_test in self._validation_loader:
                x_test, y_test = x_test.to(self._device), y_test.to(self._device)
                self._model.eval()
                z = softmax(self._model(x_test), dim=1)
                y_hat = argmax(z.data, dim=1)
                correct += (y_hat == y_test).sum().item()
                n_test += y_hat.shape[0]
                pbar.update()
            accuracy = correct / n_test
            self.validation_acc.append(accuracy)
            pbar.set_postfix({
                "Training Loss": self.training_loss[-1],
                "Validation Accuracy": accuracy,
                "Epoch": epoch + 2
            })
        pbar.set_postfix({
            "Training Loss": self.training_loss[-1],
            "Validation Accuracy": self.validation_acc[-1],
            "Epoch": n_epochs
        })

    def plot_training_stat(self):
        """
        This function plots the training statistics the model trainer collected
        throughout the training process. Namely, they are

        - Total training loss versus Iterations, and
        - Validation Accuracy versus Iterations.

        The two statistics are placed in the same plot, respectively in red and
        blue.
        """
        # Plot Total training loss versus Iterations
        fig, ax1 = subplots()
        color = 'tab:red'
        ax1.plot(self.training_loss, color=color)
        ax1.set_xlabel('Iterations', color="black")
        ax1.set_ylabel('Total Training Loss', color=color)
        ax1.set_ylim(bottom = 0)
        ax1.tick_params(axis='y', color=color)
        # Plot Validation Accuracy versus Iterations
        ax2 = ax1.twinx()
        color = 'tab:blue'
        ax2.set_ylabel('Validation Accuracy', color=color)
        ax2.plot(self.validation_acc, color=color)
        ax2.tick_params(axis='y', color=color)
        ax2.set_ylim(0, 1)
        fig.tight_layout()
        show()

    def test(self, testing_data: Dataset) -> float:
        """
        This function tests the model's performance on a given dataset.

        :param testing_data: The dataset to perform testing upon.
        :return: Model's accuracy on the given testing data.
        """
        _class = ["Hot dog", "Not hot dog"]
        j = 0
        total = 0
        print("Here are a list of inaccurately classified results:")
        for x, y in DataLoader(dataset=testing_data, batch_size=1, shuffle=True):
            x, y = x.to(self._device), y.to(self._device)
            predicted = argmax(softmax(self._model(x.to(self._device)), dim=1), dim=1)
            if predicted != y:
                j += 1
                print(f"Actual: {_class[y.item()]}\t\tPredicted: {_class[predicted.item()]}")
                imshow(x[0])
            total += 1
        return 100 - 100 * j / total

    def dump_to(self, file_name: str) -> None:
        """
        This function dumps the trained model.

        :param file_name: The directory to save state files.
        """
        save_path = join(getcwd(), file_name)
        save({"model_params": self._model.state_dict(),
              "optimiser_stats": self._optimiser.state_dict(),
              "acc": self.validation_acc,
              "loss": self.training_loss
              }, save_path)

    def load_from(self, path: str) -> None:
        """
        This function loads the dumped file back to the training framework

        :param path: The path to the dumped file.
        """
        state_dict = load(path, map_location=self._device)
        self._model.load_state_dict(state_dict["model_params"])
        self._optimiser.load_state_dict(state_dict["optimiser_stats"])
        self.validation_acc = state_dict["acc"]
        self.training_loss = state_dict["loss"]


Ahora que tenemos nuestro conjunto de herramientas de Python cargado y listo para usar, ¡comencemos a trabajar con nuestro conjunto de datos!

## Exercise 2: Get Your Tools Into Shape!<a id="Exercise2"></a>

Before we start training, let's make sure our tools and materials needed are prepared. To do that with our dataset, we'll transform the data and create a `HotDogDataset` class before training the data. In the context of machine learning, that means we'll prepare the data by loading it and set up our training framework defined above.

### Task A: Transform the Dataset<a id="Task2A"></a>

Now you might be thinking, won't the model only be trained to recognize those exact images with this dataset?

This is where image preprocessing comes in. We want to train the model with more data. One way to do this is to create copies of the image and transform those copies. This process helps the model recognize an object rather than only the picture itself. For example, we can vertically flip an image. The object in the image does not change. If we rotate that image by 5 degrees, the object in the image does not change either. It is either a hotdog or not a hotdog.

To do this with our dataset, we'll create a transformation process called "composed". This will create copies of the images that are resized, horizontally flipped and rotated. We'll also convert the images to a tensor and normalize its image channels.


In [ ]:
from torchvision import transforms

mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]
composed = transforms.Compose([
                               transforms.Resize((224, 224)),
                               transforms.RandomHorizontalFlip(),
                               transforms.RandomRotation(degrees=5),
                               transforms.ToTensor(),
                               transforms.Normalize(mean, std)
                               ])

Con la útil clase que creamos anteriormente, ahora podemos crear un objeto `dataset_full`, una instancia de la clase `HotDogDataset`.

In [ ]:
dataset_full = HotDogDataset("hotdognothotdogfull", transform = composed)

Dividamos este conjunto de datos en tres partes independientes: entrenamiento, validación y prueba. Quizás te preguntes: ¿por qué dividimos el conjunto de datos de esta manera?

Piénsalo así: el conjunto de entrenamiento es como los ejercicios de práctica que se hacen en la escuela; te ayudan a aprender y a mejorar tu capacidad para responder preguntas. El conjunto de validación equivale a un examen parcial: sirve para comprobar tus conocimientos y ofrecerte retroalimentación sobre tu desempeño hasta ese momento. Por último, el conjunto de prueba es como un examen final; evalúa qué tan bien te desempeñas, tras completar todo el proceso de aprendizaje, al enfrentarte a preguntas que quizás nunca antes habías visto.

Para entrenar el modelo, utilizaremos los datos de la primera parte del conjunto de datos. Para validar el modelo —calculando métricas como la «pérdida de validación» (validation loss)—, emplearemos los datos de la segunda parte. Finalmente, pondremos a prueba el modelo con la tercera parte del conjunto de datos, utilizando imágenes que este no ha visto anteriormente.

En este caso, destinamos el 70 % de nuestro conjunto de datos al entrenamiento, el 15 % a la validación y el 15 % restante a la prueba.

In [ ]:
from torch.utils.data import random_split
from torch import manual_seed

manual_seed(0)
training_size = int(len(dataset_full) * 0.7)
validation_size = int(len(dataset_full) * 0.15)
test_size = len(dataset_full) - training_size - validation_size
training_set, validation_set, test_set = random_split(dataset=dataset_full, lengths=(training_size, validation_size, test_size))

Feel free to adjust the training sizes as you wish! See what happens when you make the training, validation or testing sizes smaller (or larger)!

With the function we created earlier, we can preview one of the images. Let's take a look at the first one!

In [ ]:
imshow(training_set[0][0])

Podemos ver que la imagen aparece correctamente. ¡Genial, sigamos adelante!


### Tarea B: Configurar los hiperparámetros<a id="Task2B"></a>

Los hiperparámetros, como el tamaño del lote (*batch size*) o la tasa de aprendizaje (*learning rate*) de un modelo, son parámetros que pueden ajustarse para encontrar la configuración óptima para dicho modelo. Por lo general, sería necesario realizar un ajuste fino de estos hiperparámetros para determinar los mejores, ¡pero nosotros ya lo hemos hecho por ti!

Para este modelo, estableceremos el tamaño del lote, la tasa de aprendizaje y el número de épocas (*epochs*). El tamaño del lote es la cantidad de imágenes de muestra utilizadas en una única iteración del entrenamiento del modelo; la tasa de aprendizaje indica cuánto cambiará el modelo entre iteraciones; y el número de épocas corresponde a la cantidad de veces que el modelo se entrenará utilizando el conjunto de datos completo.

In [ ]:
# Batch size: train set
batch_size = 50

# Learning rate
lr = 5e-3

# Number epochs
n_epochs = 25

Ahora que ya tenemos configurados nuestros conjuntos de datos y los hiperparámetros, ¡comencemos el entrenamiento!

## Ejercicio 3: ¡Carga tu modelo!<a id="Exercise3"></a>

Comencemos a definir nuestro modelo cargándolo. Para este laboratorio, utilizaremos el modelo preentrenado ResNet18, una red de clasificación de imágenes de vanguardia desarrollada por Microsoft. Para cargar este modelo, emplearemos el siguiente código, con el parámetro `pretrained` establecido en `True`.

In [ ]:
from torchvision import models

model = models.resnet18(pretrained = True)

La red original fue diseñada para clasificar imágenes en 1000 clases distintas. En nuestro caso, solo tenemos 2 clases. Por ello, sustituiremos la capa final para reducir el número de neuronas de 1000 a 2. En consecuencia, será necesario volver a entrenar la capa final.

Para lograr esto, «congelaremos» los parámetros existentes del modelo y reemplazaremos la capa final.

In [ ]:
from torch.nn import Linear

for param in model.parameters():
    param.requires_grad = False

# Next, we set `n_classes` to the number of classes we have.
# Recall that we have two classes: "hotdog" and "not hotdog".
n_classes = dataset_full.n_classes

# Now that we have those parameters set, we can replace the output layer,
# `model.fc` of the neural network, with a `nn.Linear` object to classify
# `n_classes`'s different classes. For the first parameter, known as **in_features**,
# we input 512 because the second last hidden layer of the neural network
# has 512 neurons.
model.fc = Linear(512, n_classes)

## Ejercicio 4: ¡Busca el modelo óptimo!<a id="Exercise4"></a>

Ahora que cuentas con tu conjunto de herramientas, todo está preparado y el modelo ha sido cargado, es momento de ponerlos en práctica y emprender la búsqueda de los parámetros óptimos. El resultado será un modelo que, idealmente, presente una alta precisión; es decir, que sea capaz de determinar correctamente, en la mayoría de los casos, si una imagen corresponde o no a un «perrito caliente» (hot dog).

Para encontrar los parámetros óptimos, primero debemos definir una métrica que nos permita medir qué se considera «óptimo». En el aprendizaje automático (machine learning), esto se logra mediante la definición de una función de pérdida (loss function). Un modelo más optimizado tendrá una pérdida menor. Dado que esta red neuronal está diseñada para realizar tareas de clasificación, la «entropía cruzada» (cross entropy) —un tipo de función de pérdida— resulta la opción natural para construir el modelo.

In [ ]:
from torch.nn import CrossEntropyLoss

criterion = CrossEntropyLoss()

Tras definir la función de pérdida, debemos encontrar la manera de obtener los parámetros que minimicen dicha función. Esta tarea la llevan a cabo los optimizadores. En nuestro modelo, utilizaremos el optimizador Adam, creado por el Dr. Diederik Kingma y el profesor Jimmy Ba.

In [ ]:
from torch.optim import Adam

optimizer = Adam(model.parameters(), lr = lr)

Ahora que disponemos de todo lo necesario para entrenar el modelo, podemos configurar nuestro entorno de entrenamiento:

In [ ]:
trainer = ClassificationModelTrainer(model,
                                     training_set,
                                     validation_set,
                                     batch_size,
                                     criterion,
                                     optimizer)

¡Comencemos el entrenamiento!

Debido a nuestra capacidad de cómputo limitada, hemos realizado 25 épocas de entrenamiento en tu nombre utilizando una máquina equipada con una GPU CUDA. En el siguiente bloque de código, cargaremos el modelo entrenado y las estadísticas de entrenamiento en este entorno de ejecución de Python; a continuación, realizarás tres épocas más de entrenamiento antes de finalizar el proceso. Esto llevará unos minutos.

![image.png](data\:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAzYAAAAZCAYAAADjXbs/AAAR/UlEQVR4Ae1d643jOAy+VlLTNXCp4Bq4v8ECd5UEU8pi6lhgm/CBpCjxJVmOnZl4hgssIlsSHx8/UpLzmD+W/JcIPAmB32/X5fLj/UnSP0Dsz9tyudyWQz0Amdf78vuZ5v+6L9fLZbn9fKaSlD2PwO/lfr2cOxfmnW0jMX+uy/1Xu7Xeel9ul61z1qU+fUSU11H9yNzcHorEbDtmazMivq7Nyf5E4CQI/HESO9PMEyJw+oPNsizvPw7ekD51QSkb6DzUvEy2YA5cDubQy3jXMwQOJ5fl8tAB5ZwHG4yzeWCBtaPey9zssaV/PzHrY7OjBw+KJ3x4sMPlnPq9EMiDzfeK92He1g0bbmBgEyP/U9H8CgebZaHF9bB3P556sDksvKcRhJtHxT3Jw4PfbTsNKmc29JwHG64Tqg7WQ82Z45G2b0KgvLukeCDq0/Xtqe/VT5kKNfOw9WxKYw5KBD4WgTzYfCzeqS0RSAQSgUQgEUgEEoFEIBFIBJ6AwGkONmvvEPz3j3xSa9t/L3+Lpybuacpff5p3HOT867JL9j//4vcdnM5iz59/SV22vWL3iuxXeDr0BM6myEQgEUgEEoFEIBFIBBKBRMAhkAcbOGDkwcYRI28kAolAIpAIJAKJQCKQCCQCZ0LgRQ828L0G/eW2fMfGvptzWS5b3rGBz/7mZ77PlJtpayKQCCQCiUAikAgkAonABgQ+9mAjvljXPiYVfTk7Dzb00bUDP4p2goMNfKmx8WIDi58xVHAVYpFftnwGyN9EJnNp50+fv1R+7A5d+eW0Bx+20I9G6Idfu01KAedCgPMKP9bd+6GQSZ6hrJ6MB2Fh+3bm/YPac1oi8DwEPvlHkKj+l4f9QX4NDjajglD6yvdE3GYUnC598ic/68Isikj8y1l5sHm5gw3H1JBIEcxuUriwVy4MfvZWcMJlY6ibDsSVZ8au6B2+TYeTkT3Lsmj5wYLIvhu7nG/uxtgvGj7KTSdQ3RjbPaNbidMXHCeMt9107pStNW260j5TMdzEBdTG9j/g14gLKzyrjs6OqxMeb0R4YZ7Z/H5cBfyQOv0k9IMyP+dgAzbb+O8C4bjJw9xbU6PXc/23u5j3/ImBo/wnnT4PSZ+/P/BhmBuTPBvKiHXX/UzcvSwoc7Dm9eYdcJ/X5U04HqD3dUVojrs964rhjCfvNyquKu84R3TM1dwH611sns3NR9e2WPrwLvi9xxeFW1xT1DrU0YVjgj1WfLBBpbflDn9g0QkkMCsxSvLWQON1M5SMo40fBBjHcRHhV4cg6GgyoFs5KTfKZRO16wv+3+E7NoC1i6UDvnMDisJ1uf0wf3ATeFJJVZKsXi9U2Cd1Qnwrp5QVsW49noqWnK/7lcC5iy43l2Up+cF/uBMLl/ATuXq9L/cH/gaOttv7xbrj3NSuoR0yHit2ox91fKBbi9dXiFfLWdQt/rjpLtla0+Yrjenm6TiBsLy5je1uv0Y8E6Ye4YMQN9kkDtTaPjnr6w4DPBrHX8dPsKs9XLG5N7aT6raMsea0mW1qiOnddBnqmcwHpeiROUpAWasEhrY7ugb75ZoTjfmce4UPan3+HEteQ+vKnnXFSKr9vNr79V9PN7pUDII9kp688crn7kYBjw8Hv8SeZ5ug9XqFtWFCvotNMSQ42LwvtyIQJ1nhQWGTBUq2SQc4IQ80dKq8vr3jX+OWBbWBAwHTCwgVa3EiVoebnb9clgebBn3Q4pj2SMRTHF9gwbH84cHqFTii483ds7p5HM8DW3YtOt3F0hQuUIhji/3QLocDaxPbBq/E59hnOU7LWMlNMRHnqYV6wm4TAxdPkI+++gVd2wkDbd5rX0PZwv4jm8dwAew3PJVxLwZbvygOXPPk37AgfPgJYHvVOJFYo7eCY2RM5VqdPNEg+a5Go9+wkRb6jW7EodZo7ZPqqwdpNgdk3pb3wjP3TpG8r/gN83ndEHbZMWo+ryftUMBWxK8gV/uixlnZ1jdcO1mnzyGFC6+ZSsHkBdoxsFOJ8T6hHdZ2noOyZ/HiSZ3XYC/hdBvMHBdBdMcmhWfkj5Gt36niGs3xangquZXj/af00TpkZTS/JjjcgbPeBr8wH0su1Q5uyPy4uA1q17YqV8ppuOBDN9Ar80DhTmtQrXWmZoDUWDfZ2zCCkdE9tsu8Ypw1Z/16ZeaISxirYtjhG04Z9bF/gd9C3YYm4alxadMBS7BbrkF67JgHICmORzncXe/LO7z5UXJAYdTMWG8hZoJHcD2JEdqnOEbqgoNNswMnGQVWUHUcx1mgG5Gd00C2wCDSDvOEowHADCa95sFG4xFspDaQpTFAP52wsVfjOEaSL5M6u3JFQeqOKUbYQgXjHeeswaNrTDZdDGk4FAPJzVYcdNFYqKB0OI7+KDmxMdYvHoXzJdbcUf6gqH+asmK3XbQQe+CRwQBxMYXe/RFTk/ezsqsPxzb2cYF8IS4ZDDf45RZHdrHLMx5QFpeAR12ZberOFnHb8poPt+1jxmbcr/tyE3+IsMdVvO/8IlmNd0Y2exTixrzj/JSxg4nmGjnOY1nw6BVs6Y23dtJ1q0Gjuf3N+ciabh9i07PTz8Iaw3le8tvFvEyLY+Zlzt2xmFF8mu735Sb5gfEy9QgUhVxoFoQ2W4ysjAkOz+RfOMb6oXi4xuHmV6/VdFo8YYblpZaCWDEXdBd9UkGtOYbT6IdYMwzGv99uy/0XCzW5yPuHjm7k6AwXWLx4tfEnH/2BTkzRzeIX8bLER9oiRjfsxU3RRN0KQ9G5uRnFtwlhP2sNUryz+Hu/aH6Qb6CCY804KNnNhqmW4wl8MuiOb3zwvrb6YASijWyD6Hv8YMOOQZBqUWhAIxEvtAHyyk1CCIOoCXJ0YeYgsaP6NQ82Go+jDjYtnhAXH0cROOSA+ZJ9uddsi5JE62gS9f2hbuTiCl8C8jddQaty2vYxd8k+8A0KXlTQXDG2otauA794CuLhCiQtWvEDgxW7QVfNZV6ceA5r7b22WIV5v0t2T+f8fVc7tnCBbUd1Bg/uqzwHfpsxxcyIH9jV5Rn717DlO/yKWDsOcO8Rr+CLyWkQW/xtG9CY/9WCjo9xTnv8QuxCmR4rpQPnyDrR8a8abhveNh4R5iPzAweRrt4CHWHKsnkz2uoo1Xf/8AJmlLq0heMwDWzFJ68SH7aAbO/387jtr8hhtjWMqZTZwX9lnuJAEec4tSIjOjw5GdLUnh57uHbjVjgc6NC3NEbW95CnVQDFWeZ17YKG4jPc0LqIQ5I/3hcpT9u2otvER2NPc3v5UfUwx+s6F+1HpIWy3XSMc3ggE30I6qlUs6ldcl2+aygevFS/q0wRL8TC2KowXonHkAsNKxWTcK3y9QprgvBjVBu9j+TsYwcbAFIaWZ1sQMvgaxK2RZAcgCItkwEMAzn6HjqgAliKeynG+R0biQe1ZQyQHDJmlez9hiWNvW4zichKX+usLYqhSabKnToMG1aXva6jp4qFT546v9dQSS4HtaRtCwDJb9c0HvnNi7YUMdNe8QvxsPHEYqU/EtFUrdjNc+UTsy4GTSq1VvJ+l2yra+/1Fi4AZrIOmesNftkaWL1Ywxh02DjXyQ13WEAs/+qwhxvEGSd3zWbeXKt6bfK++6DEYAzbJ/tREPAntMHnoa4b5E+tUxg/b1cfLm8bj9V6yl0bO7SZ63Sgt/LJrK+sZOIVa06XL7EANafYUDFyUzqccOMmb4gYRBiibYpHMh+LjpALTX9PrvLRydC5RRs0HbOQl00ttvwYz1EY2Mb5/sh+o6ZdhpxrdiOevTUJMQjwZelWdnSwGXFP8b/kAduyprvUFKpFwMHmE5vXe0X8gEPSNudLbzYf+hkX4r+SVaYOsV15p2ygfdDluSIHe96I3I38lzmwFg83H2QzRtKKcRsxk3FRudDmtvxo96DlfaT+zQcbOpVrUknhPrgGfAZEgAjzVZHJg015evaZP/dMcVMn7rrASAKXROcCpXmnr0TMuSMm7KRulGc/FsWS9avkqO7pXAW20kiyTfM1TmqfCx1d9vaEX+iPKQjSPl94V+xGnTKu0RM6a2i79r6SPlyIdspuWo5pTXMBalXlPG9IxYK8wa+Y570NevOzO68NoVbhjDuE2HGbrsVCKOehLr0GyG7kguRmZ3wcB59LIQahTMG5YpDWQf0yptvw8rax31E+RvfU+MHmTGNIcZB2Y1tiXDYE7SN8rGnlFXEM8n7FNl3/VnQMu5ljndgpOzr4h1xoSjUH6L7jlJGh8Y/z1MloKmvLjyE/NX7yXgeHmfW1csDUKvHQI8KiGmsPKq2DWrx3q/dNPFx/HQiW4bu/0m9ti5Elp5Z2HQ96FB4ku5sfWMd1vaqyAj36lowN95A+VTuiPOLhxXdtc+3c0fBckcK8jwLjABO9txdjpVBuu1jL8YRPNx5FBuaYym/q8HbLgz8b0B8LPdsPNi5I5EQNMgImnh4qAIXzopDEya+LLTrb22Rc8qNojkTlY4CVBoC3WQhr32TDE64QWBWZnjBKQpXcggO9WXzf6ca5c4ea0VuZLN+9DmwjLjZ+4nWALSZuBxsro+qf9Kunk+VERcPq1DJKfKofUUHnBT7AfZj3k7KLDMURduio14JvrVdFLuEl6laoT9Qv7J/0q/MUilSY+in1Djgoh43kzPnlpY1kxu+WNBma9wWjyQXMfbylh12IDemSsUWOcw4Cvyq/m73zLRt/MdPxahBXmKbWRSGnNHVu+n57J8p3PYbj0OoW9ju7aRPRxQnHGxliQy2x1/r7V2j79bpcDUcsBuSj173GR8WBYoaWTbGSh8IZDmsZsX8gR27mYRTOk74qLqxwOFbDXoU/yKR8CeItReLYXo4EdqpP2wzzy+RDsUPW+qFuNJLy72o+zSPtj9slvlwHyh7WchX1i0MgySp5U+dy7moeKoyVEVa36iwXndyMhqp7niuy2/Je40t2NW6SrHa9UgdcrCk27TtU0hLfJqz1YbOOsjXGXteBm96xYZDNqV+RvQSrHDQsQaho8/xmPDjjgEMZbQzZDDZo4lAxYJn2NQ82n3Gw6cWE+WD7ZewhzjbxBF9d046lxPA8oMSyHNZccsKjG5hMlpdtoPJtkBs1LrIwsu/BW7fb/Cr+K/3GxlBvb57GzcYLJZcFKezDhY9jYrGbkM0PTTr+NM+2tLRetRALMYw7c1d0iWZUvLV8jYuuk8wFPYYXSsatcdVyXhhSn4CyTHh1cmFCiUnYpwUGV2S/w2QlN/hBAtt2e7uLTavGi8e0DY7HWK4bKu/qgy7mGsmW9moMY91yfACCuBXFs8XL+q0wV7kBsRbzaj1gDsAr+yTU95olJyuWjIvKffZd60WR1jaVf9bnYD4I2cMz1q/sBaFa9/Xtrj7uMsOFPiaMB8cCdAnMDaaawxwIKUN+BFjbzTZIPmjbJaYkU3JSc5h1B6+9vER8+77ZQyzXQrZb2iL7bj/BT2E76FHc0TZqn8ufEzExl/JBv9QN0rB/oENrlFc6JlZulR3otDy0uUvcFzgItdrnlt9WP4+z94WooGn4Z/bjLJPj6GOjMWk1uKnqxsPF2nChifAtk1vVPskFrgnoU4wtCEYf5byibfiOjbfoo+5AwLQzLkhcvIvj+R2bljRMFFlIcdF9qCA8K+YbEuFZJozk9haJ0Zzs24lAKdRBodop+ITTD8qPsohsWzBPCNeMyW4x5g252PTNyMkxHoHkmcck7xyOAGy01b7mcA2fJBA38nrPu9eS3qZ/r9xXmt/zMQ82cDDKv2PzSlx9DVvyYPOhcagPLvJQcxDu7WleHmoIUuSYebjz+BPgg8J0ejHJs9OH8CwO4Ob/qz2E4HdNjj3UQEh7m/6zhHvGzp6PebDJg80Mf77fmPIEkt/9ys3h96NAevzVEGibcM5r//GMr+Zz+pMInByB+rGk4zf/J0dmaH5v0z+cdJJO9RG54GHoix5sPLoYJPXxM/nRq/yOTV2oBUZf8i1bT428kwgkAolAIpAIJAKJQCKQCIx/FS3xSQQSgUQgEUgEEoFEIBFIBBKBROAMCJzmHZszgJk2JgKJQCKQCCQCiUAikAgkAonA5yCQB5vPwT21JgKJQCKQCCQCiUAikAgkAonAgQjkweZAMFNUIpAIJAKJQCKQCCQCiUAikAh8DgL/A2+TS0uZRYosAAAAAElFTkSuQmCC)

In [ ]:
import os
# Activamos el interruptor maestro del sistema ANTES de que PyTorch se cargue de verdad
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
print("¡Filtro de seguridad desactivado en el sistema operativo de Colab!")


In [ ]:
# Tu código original ahora funcionará sin el bloqueo de seguridad de PyTorch
trainer.load_from("/content/twenty-five-iters.pt")
trainer.train_model(n_epochs=2)



¡Tu modelo ha sido entrenado!

En los últimos minutos, tu modelo ha aprendido gradualmente a distinguir si una imagen contiene un perrito caliente o no.

Durante el proceso, hemos medido qué tan bien ha aprendido. ¡Echemos un vistazo!

In [ ]:
trainer.plot_training_stat()

En este gráfico aparecen dos curvas: una representa la pérdida durante el entrenamiento y la otra, la precisión en la validación. La pérdida en el entrenamiento mide qué tan familiarizado está el modelo con los datos que le hemos proporcionado para aprender. La precisión en la validación mide su capacidad para clasificar correctamente si una imagen que nunca antes ha visto contiene o no un perrito caliente.

Ahora es el momento de someter al modelo a su examen final: ¿logrará superar con éxito el conjunto de prueba?

In [ ]:
accuracy = trainer.test(test_set)

Las imágenes que identificó incorrectamente se muestran en el resultado anterior. Para ver qué puntuación obtuvo en el conjunto de prueba, podemos imprimir su precisión:

In [ ]:
print(f"The model reached an accuracy rate of {accuracy:.2f}% on images it has never seen before.")

Has entrenado tu modelo con éxito y has determinado su precisión. ¡Ahora, veamos qué has aprendido durante el proceso!

## Ejercicio 5: ¡Recapitulación!

¿Qué has aprendido de este proyecto? ¡Echemos un vistazo!

### Pregunta 1: Basándote en las curvas de entrenamiento, ¿consideras que hemos logrado una construcción de modelo válida? ¿Por qué sí o por qué no?

<details><summary>Haz clic aquí para obtener una pista:</summary>
Observa la precisión en la validación. ¿Notas algún patrón (o la ausencia del mismo)?
</details>

### Pregunta 2: Basándote en las curvas de entrenamiento, ¿crees que cuanto menor sea la pérdida de entrenamiento, mejor será el rendimiento del modelo?

<details><summary>Haz clic aquí para obtener una pista:</summary>
Al observar el gráfico que creamos con la pérdida de entrenamiento y la precisión de validación, ¿notas alguna tendencia?
</details>


### Pregunta 3: Si quisieras entrenar un modelo para clasificar si una imagen contiene un gato o no, ¿cómo lo harías?

<details><summary>Haz clic aquí para obtener una pista:</summary>
Piensa en el proceso que seguimos para detectar si una imagen contiene un perrito caliente (hot dog) o no. ¿Qué cambiarías?
</details>
